In [ ]:
import os
from langchain_community.document_loaders import TextLoader

loader_text=TextLoader("../data/text/New Text Document.txt")
document_text=loader_text.load()

In [ ]:
print(f"📄 Loaded {len(document_text)} document")
print(f"Content preview: {document_text[0].page_content[:100]}...")
print(f"Metadata: {document_text[0].metadata}")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=[
        "\n\n",
        ".",
        " ",
    ],
    chunk_size=400,
    chunk_overlap=50,
)

chunks = splitter.split_text(text)

print(f"Chunks: {len(chunks)}")
print(chunks[0])

In [ ]:
print(recursive_chunks[0])
print("-----------------")
print(recursive_chunks[1])
print("------------------")
print(recursive_chunks[2])


In [90]:
# ============================================
# 🚀 FINAL GENERAL PDF PROCESSOR (BEST VERSION)
# ============================================

import os
import re
import fitz
import pdfplumber

from typing import List, Dict
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ============================================
# CORE PROCESSOR
# ============================================

class GeneralPDFProcessor:

    def __init__(self, chunk_size=500, chunk_overlap=80):
        self.text_splitter = RecursiveCharacterTextSplitter(
            separators=["\n\n", "\n", ".", " "],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

    # ============================================
    # MAIN
    # ============================================
    def process(self, pdf_path: str) -> List[Document]:

        if not os.path.isfile(pdf_path):
            raise ValueError(f"❌ Invalid file path: {pdf_path}")

        text_docs = self._process_text(pdf_path)
        table_docs = self._process_tables(pdf_path)

        print(f"   ✅ Text chunks: {len(text_docs)}")
        print(f"   📊 Table chunks: {len(table_docs)}")

        return text_docs + table_docs

    # ============================================
    # TEXT PROCESSING (🔥 FIXED)
    # ============================================
    def _process_text(self, pdf_path: str) -> List[Document]:

        loader = PyMuPDFLoader(pdf_path)
        pages = loader.load()

        docs = []

        last_section_id = None
        last_title = None

        for page_num, page in enumerate(pages):

            text = page.page_content
            if not text:
                continue

            text = self._clean_text(text)

            if len(text.strip()) < 40:
                continue

            sections = self._split_sections(text)

            for section in sections:

                if self._is_noise(section):
                    continue

                section_id = self._get_section_id(section)
                section_title = self._get_title(section)

                # 🔥 FIX: inherit previous section
                if section_id == "unknown":
                    section_id = last_section_id
                    section_title = last_title
                else:
                    last_section_id = section_id
                    last_title = section_title

                hierarchy = self._get_hierarchy(section_id, section_title)

                chunks = self.text_splitter.create_documents(
                    texts=[section],
                    metadatas=[{
                        **page.metadata,
                        "page": page_num + 1,
                        "type": "text",
                        "section_id": section_id,
                        "section_title": section_title,
                        "hierarchy": hierarchy
                    }]
                )

                docs.extend(chunks)

        return docs

    # ============================================
    # TABLE PROCESSING (IMPROVED)
    # ============================================
    def _process_tables(self, pdf_path: str) -> List[Document]:

        docs = []

        # -------- pdfplumber --------
        try:
            with pdfplumber.open(pdf_path) as pdf:
                for i, page in enumerate(pdf.pages):

                    tables = page.extract_tables()

                    for table in tables:
                        if not table:
                            continue

                        text_table = "\n".join(
                            [" | ".join([str(c) if c else "" for c in row]) for row in table]
                        )

                        docs.append(Document(
                            page_content=text_table,
                            metadata={
                                "page": i + 1,
                                "type": "table",
                                "method": "pdfplumber"
                            }
                        ))
        except Exception as e:
            print("⚠️ pdfplumber failed:", e)

        # -------- fallback --------
        try:
            doc = fitz.open(pdf_path)

            for page_num, page in enumerate(doc):

                blocks = page.get_text("blocks")

                for b in blocks:
                    text = b[4]

                    if self._looks_like_table(text):
                        docs.append(Document(
                            page_content=text,
                            metadata={
                                "page": page_num + 1,
                                "type": "table",
                                "method": "fallback"
                            }
                        ))
        except Exception as e:
            print("⚠️ fallback failed:", e)

        return docs

    # ============================================
    # HELPERS
    # ============================================

    def _clean_text(self, text: str) -> str:
        if not text:
            return ""
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        return text.strip()

    def _split_sections(self, text: str) -> List[str]:
        sections = re.split(
            r'\n(?=\d+(\.\d+)*\s+[A-Z]|[A-Z][A-Z\s]{5,}|Chapter\s+\d+)',
            text
        )
        return [s.strip() for s in sections if s and s.strip()]

    def _get_title(self, text: str) -> str:
        lines = [l.strip() for l in text.split("\n") if l.strip()]

        if not lines:
            return "UNKNOWN"

        if re.match(r'^\d+(\.\d+)*$', lines[0]) and len(lines) > 1:
            return lines[1]

        match = re.match(r'^(\d+(\.\d+)*)\s+(.*)', lines[0])
        if match:
            return match.group(3)

        if lines[0].lower().startswith("chapter") and len(lines) > 1:
            return lines[1]

        if lines[0].isupper():
            return lines[0]

        return lines[0][:60]

    def _get_section_id(self, text: str) -> str:
        first = text.split("\n")[0].strip()

        match = re.match(r'^(\d+(\.\d+)*)', first)
        if match:
            return match.group(1)

        if first.lower().startswith("chapter"):
            return first

        return "unknown"

    # 🔥 FIXED HIERARCHY
    def _get_hierarchy(self, section_id: str, title: str) -> Dict:
        if not section_id:
            return {"level": 0, "path": [title]}

        return {
            "level": section_id.count(".") + 1,
            "path": [section_id, title]
        }

    def _is_noise(self, text: str) -> bool:
        if not text:
            return True

        t = text.lower().strip()

        if len(t) < 40:
            return True

        if "contents" in t:
            return True

        if re.search(r'\.{3,}', t):
            return True

        return False

    # 🔥 BETTER TABLE DETECTION
    def _looks_like_table(self, text: str) -> bool:
        if not text:
            return False

        if "|" in text:
            return True

        lines = text.split("\n")

        numeric_lines = sum(
            1 for l in lines if len(re.findall(r'\d+', l)) >= 3
        )

        return numeric_lines >= 3


# ============================================
# 📂 PROCESS FOLDER
# ============================================

def process_pdf_folder(folder_path: str) -> List[Document]:

    processor = GeneralPDFProcessor()
    all_docs = []

    if not os.path.exists(folder_path):
        raise ValueError(f"❌ Folder not found: {folder_path}")

    pdfs = [f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")]

    print(f"📂 Found {len(pdfs)} PDFs")

    for pdf in pdfs:

        path = os.path.join(folder_path, pdf)
        print(f"\n🚀 Processing: {pdf}")

        try:
            docs = processor.process(path)

            for d in docs:
                d.metadata["source_file"] = pdf

            all_docs.extend(docs)

        except Exception as e:
            print(f"❌ Error with {pdf}: {e}")

    print(f"\n🔥 TOTAL DOCUMENTS: {len(all_docs)}")

    return all_docs


# ============================================
# 🚀 RUN
# ============================================

if __name__ == "__main__":

    folder_path = "../data/pdf/"

    docs = process_pdf_folder(folder_path)

    # Preview
    for i, d in enumerate(docs[:10]):
        print("\n" + "="*60)
        print(f"Doc {i+1}")
        print(f"File: {d.metadata.get('source_file')}")
        print(f"Type: {d.metadata.get('type')}")
        print(f"Page: {d.metadata.get('page')}")
        print(f"Section: {d.metadata.get('section_id')}")
        print(f"Title: {d.metadata.get('section_title')}")
        print(f"Hierarchy: {d.metadata.get('hierarchy')}")
        print("-"*60)
        print(d.page_content[:])

📂 Found 7 PDFs

🚀 Processing: Chapter 3 - Implementation and Realization (7).pdf
   ✅ Text chunks: 30
   📊 Table chunks: 1

🚀 Processing: EFFIA Parking Policy.pdf
   ✅ Text chunks: 34
   📊 Table chunks: 0

🚀 Processing: EFFIA User Guide.pdf
   ✅ Text chunks: 26
   📊 Table chunks: 0

🚀 Processing: Ennocé Examen.pdf
   ✅ Text chunks: 7
   📊 Table chunks: 0

🚀 Processing: projet_d_integration__Copy__ff.pdf
   ✅ Text chunks: 191
   📊 Table chunks: 15

🚀 Processing: Rapport Microservices.pdf
   ✅ Text chunks: 30
   📊 Table chunks: 1

🚀 Processing: Rapport-PFE-15072022-001 (2) (1).pdf
   ✅ Text chunks: 181
   📊 Table chunks: 21

🔥 TOTAL DOCUMENTS: 537

Doc 1
File: Chapter 3 - Implementation and Realization (7).pdf
Type: text
Page: 1
Section: Chapter 1
Title: Implementation and Realization
Hierarchy: {'level': 1, 'path': ['Chapter 1', 'Implementation and Realization']}
------------------------------------------------------------
Chapter 1
Implementation and Realization
Introduction
After defi

In [47]:

# ============================
# 🚀 RUN + DISPLAY
# ============================

try:
    preprocessor = SmartParsingFile()

    smart_chunks = preprocessor.process_pdf(
        "../data/pdf/EFFIA Parking Policy.pdf"
    )

    print(f"\n✅ Processed into {len(smart_chunks)} smart chunks")

    for i, chunk in enumerate(smart_chunks[:]):
        print("\n" + "=" * 60)
        print(f"📦 Chunk {i+1}")
        print(f"📄 Page: {chunk.metadata.get('page')}")
        print(f"🏷️ Section: {chunk.metadata.get('section')}")
        print(f"🔢 Section ID: {chunk.metadata.get('section_id')}")
        print("-" * 60)
        print(chunk.page_content[:300])

    if smart_chunks:
        print("\n📊 Sample Metadata:")
        for k, v in smart_chunks[0].metadata.items():
            print(f"{k}: {v}")

except Exception as e:
    print(f"❌ Processing error: {e}")


✅ Processed into 37 smart chunks

📦 Chunk 1
📄 Page: 1
🏷️ Section: EFFIA Online Parking Reservation
🔢 Section ID: unknown
------------------------------------------------------------
EFFIA Online Parking Reservation
Policy
Standard Operating Procedure & Guidelines

📦 Chunk 2
📄 Page: 1
🏷️ Section: EFFIA Operations Management
🔢 Section ID: unknown
------------------------------------------------------------
EFFIA Operations Management
May 5, 2026
Version 1.1

📦 Chunk 3
📄 Page: 2
🏷️ Section: 3.1
🔢 Section ID: 3.1
------------------------------------------------------------
3.1
Priority Tiers . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
1

📦 Chunk 4
📄 Page: 2
🏷️ Section: 3.2
🔢 Section ID: 3.2
------------------------------------------------------------
3.2
Electric Vehicle (EV) Charging Stations (EFFIA Park & Charge) . . . . . . . . . . . . .
1

📦 Chunk 5
📄 Page: 2
🏷️ Section: 3.3
🔢 Section ID: 3.3
-----------------------------------------------------